# AIML339 Galaxy Morphology Classification Comparison Project

donaldseth - 300665568

In [1]:
# Imports

import pandas as pd
import numpy as np

from zipfile import ZipFile
from PIL import Image
from io import BytesIO

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

KeyboardInterrupt: 

In [ ]:
# Paths

LABELS_PATH = "training_solutions_rev1.csv"
IMAGES_PATH = "images_training_rev1.zip"

In [ ]:
# Load Labels

labels = pd.read_csv(LABELS_PATH)

print("Total galaxies:", len(labels))
print("Number of classes:", len(labels.columns) - 1)

Total galaxies: 61578
Number of classes: 37


In [ ]:
# Labels Subset

N_GALAXIES = 5000

labels_subset = labels.sample(
    n=N_GALAXIES,
    random_state=42
).copy()

labels_subset["filename"] = (
    labels_subset["GalaxyID"].astype(str)
)

labels_subset.head()

,GalaxyID,Class1.1,Class1.2,Class1.3,Class2.1,Class2.2,Class3.1,Class3.2,Class4.1,Class4.2,...,Class10.1,Class10.2,Class10.3,Class11.1,Class11.2,Class11.3,Class11.4,Class11.5,Class11.6,filename
30888,553402,0.018765,0.955090,0.026146,0.078768,0.876322,0.118995,0.757327,0.590601,0.285720,...,0.000000,0.136383,0.454219,0.193254,0.289966,0.000000,0.0,0.000000,0.107381,553402
23964,449744,0.806705,0.140380,0.052915,0.000000,0.140380,0.000000,0.140380,0.000000,0.140380,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,449744
53475,879640,0.662354,0.337646,0.000000,0.197076,0.140570,0.000000,0.140570,0.044101,0.096468,...,0.044101,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.044101,879640
16284,340485,0.092952,0.907048,0.000000,0.058283,0.848765,0.042052,0.806713,0.642360,0.206404,...,0.496498,0.134098,0.011765,0.000000,0.373704,0.010232,0.0,0.013647,0.244777,340485
60574,985545,0.144641,0.848689,0.006670,0.028017,0.820672,0.484659,0.336014,0.267777,0.552895,...,0.000000,0.219558,0.048219,0.000000,0.195449,0.000000,0.0,0.000000,0.072328,985545


In [ ]:
# Collect from .zip file

with ZipFile(IMAGES_PATH, "r") as zip_file:
    
    images = []

    for galaxy_id in labels_subset["filename"]:
        filename = f"images_training_rev1/{galaxy_id}.jpg"
        image_data = zip_file.read(filename)

        image = Image.open(BytesIO(image_data))
        images.append(image.copy())

print("Images collected:", len(images))

Images collected: 5000


In [ ]:
# Train, Validation, Test Split

train_val, test = train_test_split(
    labels_subset,
    test_size=0.20,
    random_state=42
)

train, validation = train_test_split(
    train_val,
    test_size=0.20,
    random_state=42
)

print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Training: 3200
Validation: 800
Test: 1000


In [ ]:
# Image Transformation

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

#transform = transforms.Compose([
#    transforms.Grayscale(num_output_channels=1),
#    transforms.Resize((64, 64)),
#    transforms.ToTensor()
#])

In [ ]:
class GalaxyDataset(Dataset):
    
    def __init__(self, data, transform=None):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.zip_file = ZipFile(IMAGES_PATH, "r")
        
        self.label_columns = [
            column for column in self.data.columns
            if column not in ["GalaxyID", "filename"]
        ]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        
        row = self.data.iloc[index]
        
        # Construct the actual filename inside the ZIP
        filename = f"images_training_rev1/{row['filename']}.jpg"
        
        # Read the image directly from the ZIP
        image_data = self.zip_file.read(filename)
        image = Image.open(BytesIO(image_data)).convert("RGB")
        
        # Apply preprocessing
        if self.transform:
            image = self.transform(image)
        
        # Get the 37 Galaxy Zoo probabilities
        labels = torch.tensor(
            row[self.label_columns].values.astype(np.float32)
        )
        
        return image, labels

In [ ]:
# Dataset Creation

train_dataset = GalaxyDataset(train, transform)
validation_dataset = GalaxyDataset(validation, transform)
test_dataset = GalaxyDataset(test, transform)

print("Datasets created")

Datasets created


In [ ]:
# DataLoaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
images, labels_batch = next(iter(train_loader))

print("Image shape:", images.shape)
print("Label shape:", labels_batch.shape)

Image shape: torch.Size([32, 3, 64, 64])
Label shape: torch.Size([32, 37])


In [ ]:
# CNN

class GalaxyCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # nn.Conv2d(1, 16, kernel_size=3, padding=1)
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 37)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# GPU

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

Using: cuda
NVIDIA GeForce RTX 5070 Laptop GPU


In [ ]:
# Model

model = GalaxyCNN().to(device)

print(model)

GalaxyCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=37, bias=True)
  )
)


In [ ]:
# Loss and optimizer

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# Training

epochs = 10 # starting with 10 to make sure it works

for epoch in range(epochs):
    
    model.train()
    running_loss = 0.0
    
    for images, labels_batch in train_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(images)
        loss = criterion(predictions, labels_batch)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    average_loss = running_loss / len(train_loader)
    
    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 0.0279
Epoch [2/10], Loss: 0.0250
Epoch [3/10], Loss: 0.0226
Epoch [4/10], Loss: 0.0212
Epoch [5/10], Loss: 0.0194
Epoch [6/10], Loss: 0.0182
Epoch [7/10], Loss: 0.0176
Epoch [8/10], Loss: 0.0166
Epoch [9/10], Loss: 0.0162
Epoch [10/10], Loss: 0.0153


In [ ]:
# Validation

model.eval()

validation_loss = 0.0

with torch.no_grad():
    
    for images, labels_batch in validation_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        predictions = model(images)
        loss = criterion(predictions, labels_batch)
        
        validation_loss += loss.item()

validation_loss /= len(validation_loader)

print(f"Validation Loss: {validation_loss:.4f}")

Validation Loss: 0.0192


In [ ]:
# Test

model.eval()

test_loss = 0.0

with torch.no_grad():
    
    for images, labels_batch in test_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        predictions = model(images)
        loss = criterion(predictions, labels_batch)
        
        test_loss += loss.item()

test_loss /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.0182


In [ ]:
# Look at one galaxy

image, true_labels = test_dataset[0]

model.eval()

with torch.no_grad():
    prediction = model(
        image.unsqueeze(0).to(device)
    )

prediction = prediction.cpu().numpy()[0]
true_labels = true_labels.numpy()

print("True probabilities:")
print(true_labels)

print("\nPredicted probabilities:")
print(prediction)

True probabilities:
[0.127217   0.847839   0.024944   0.36012724 0.48771176 0.05610148 0.43161026
 0.21759942 0.27011234 0.04478072 0.35924703 0.         0.08368354 0.295699
 0.704301   0.         0.02694125 0.10027575 0.         0.08087368 0.07423228
 0.11761516 0.02297818 0.         0.         0.118076   0.03941593 0.20263532
 0.1229798  0.09461963 0.         0.1424334  0.         0.         0.
 0.         0.07516602]

Predicted probabilities:
[ 0.20433518  0.78136945 -0.00384194  0.43673253  0.31920156  0.05396172
  0.2608067   0.18770504  0.13642862  0.0209236   0.15990907  0.11760258
 -0.00679456  0.12961237  0.8283274  -0.02222778  0.05524473  0.15683219
  0.00918063  0.01409188  0.01632525  0.01115402  0.02784245  0.02992276
  0.01510477  0.25626743  0.05455816  0.1571239   0.07203628  0.06713492
  0.03634062  0.02131246  0.06811078  0.00757829 -0.00191103  0.02102032
  0.06534243]


In [ ]:
print(labels_subset.columns.tolist())

['GalaxyID', 'Class1.1', 'Class1.2', 'Class1.3', 'Class2.1', 'Class2.2', 'Class3.1', 'Class3.2', 'Class4.1', 'Class4.2', 'Class5.1', 'Class5.2', 'Class5.3', 'Class5.4', 'Class6.1', 'Class6.2', 'Class7.1', 'Class7.2', 'Class7.3', 'Class8.1', 'Class8.2', 'Class8.3', 'Class8.4', 'Class8.5', 'Class8.6', 'Class8.7', 'Class9.1', 'Class9.2', 'Class9.3', 'Class10.1', 'Class10.2', 'Class10.3', 'Class11.1', 'Class11.2', 'Class11.3', 'Class11.4', 'Class11.5', 'Class11.6', 'filename']
